In [1]:
import pandas as pd
import duckdb as ddb
import os

In [11]:
df = pd.read_excel('/home/roman/Downloads/GAAP_Taxonomy.xlsx', sheet_name='Presentation')[['definition', 'prefix', 'name','label' ,'parent', 'depth', 'order', 'priority']]

In [5]:
df['parentName'] = df['parent'].str.split(':').str[1]
df['parentPrefix'] = df['parent'].str.split(':').str[0]
#df.drop(columns=['parent'], inplace=True)
df['name'] = df['name'].str.strip()

In [ ]:
from collections import defaultdict

normalized_data = []

grouped = df.groupby('definition')

for def_name, def_df in grouped:
    children = defaultdict(list)
    for _, row in def_df.iterrows():
        if pd.notna(row['parentName']) and pd.notna(row['parentPrefix']):
            children[(row['parentPrefix'], row['parentName'])].append((row['prefix'], row['name']))
    
    has_children = set(children.keys())
    top_level_nodes = set(
        (row['prefix'], row['name'])
        for _, row in def_df.iterrows()
        if pd.isna(row['parentName']) and pd.isna(row['parentPrefix'])
    )
    node_weights = {
        (row['prefix'], row['name']): row['priority']
        for _, row in def_df.iterrows()
    }
    
    def get_subtree(node, rel_depth=0):
        result = [(node, rel_depth)]
        if node in children:
            for child in children[node]:
                result.extend(get_subtree(child, rel_depth + 1))
        return result
    
    for _, node in def_df[['prefix', 'name']].drop_duplicates().iterrows():
        node_tuple = (node['prefix'], node['name'])
        subtree = get_subtree(node_tuple)
        for desc, rel_dep in subtree:
            normalized_data.append({
                'definition': def_name,
                'ancestor': f"{node_tuple[0]}:{node_tuple[1]}",
                'descendant': f"{desc[0]}:{desc[1]}",
                'relative_depth': rel_dep,
                'priority': node_weights.get(desc),
                'highestParent': node_tuple in top_level_nodes,
                'lowestChild': desc not in has_children
            })

normalized_df = pd.DataFrame(normalized_data)

In [40]:
normalized_df.to_excel("Normalized_GAAP_Taxonomy.xlsx", index=False)

In [ ]:
normalized_df['ancestorName']  = normalized_df['ancestor'].str.split(':').str[1]
normalized_df['ancestorPrefix'] = normalized_df['ancestor'].str.split(':').str[0]
normalized_df['descendantName'] = normalized_df['descendant'].str.split(':').str[1]
normalized_df['descendantPrefix'] = normalized_df['descendant'].str.split(':').str[0]

TypeError: list indices must be integers or slices, not str

In [7]:
db_directory = "/home/roman/Documents/EDGAR_Analytics/Data"
db_path = os.path.join(db_directory, "secFilingsDb.duckdb")

conn = ddb.connect(db_path)

In [8]:
df = conn.execute("""SELECT
    cik,
    accessionNumber,
    startDate,
    endDate,
    name,
    units,
    prefix,
    COUNT(*) AS occurrence_count
FROM financialData
GROUP BY accessionNumber, name, startDate, endDate, units,cik, prefix
HAVING COUNT(*) > 1
ORDER BY accessionNumber, occurrence_count DESC;""").fetch_df()

RuntimeError: Query interrupted

In [9]:
conn.register('presentation_df', normalized_df)
conn.execute("CREATE TABLE presentationTaxonomy AS SELECT * FROM presentation_df")

In [ ]:

import os
from curl_cffi import requests
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import duckdb as ddb
from dotenv import load_dotenv

def download_cal_xmls(conn):
    load_dotenv()

    PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT") or Path(__file__).resolve().parent.parent)
    DATA_DIR = PROJECT_ROOT / "Data"

    for d in [DATA_DIR]:
        d.mkdir(parents=True, exist_ok=True)

    save_path = os.path.join(DATA_DIR, "calXMLs/")
    xml_taxonomy_folder = Path(save_path)
    xml_taxonomy_folder.mkdir(parents=True, exist_ok=True)

    records = set()
    for f in xml_taxonomy_folder.iterdir():
        if f.suffix in (".xml", ".xsd"):
            parts = f.stem.split("_")
            if len(parts) >= 2:
                cik, accession_number = parts[0], parts[1]
                records.add((cik, accession_number))


    df_db = conn.execute("""SELECT DISTINCT cik, accessionNumber from FinancialData
        """).fetch_df()
    conn.close()

    if records:
        df_folder = pd.DataFrame(records, columns=["cik", "accessionNumber"])
        df_folder['cik'] = df_folder['cik'].astype("int32")
        df = df_db.merge(df_folder, on=["cik", "accessionNumber"], how="left", indicator=True)
        df= df[df["_merge"] == "left_only"].drop(columns="_merge")
    else:
        df = df_db
    if not df.empty:
        df['url'] = df.apply(lambda row: f"https://www.sec.gov/Archives/edgar/data/{row['cik']}/{row['accessionNumber'].replace('-', '')}/FilingSummary.xml", axis=1)
        df['url_no_file'] = df.apply(lambda row: f"https://www.sec.gov/Archives/edgar/data/{row['cik']}/{row['accessionNumber'].replace('-', '')}/", axis=1)

        def download_cal_xmls(url, urlNoFile, cik, accessionNumber, savePath):
            cal_link = None
            cal_file = None
            os.makedirs(savePath, exist_ok=True)
            try:
                response = requests.get(url, impersonate="chrome101", stream=False,timeout=30)
                root = ET.fromstring(response.text)
                response.raise_for_status() 
                
            except Exception as e:
                response = requests.get(urlNoFile, impersonate="chrome101", stream=False,timeout=30)
                response.raise_for_status() 
                soup = BeautifulSoup(response.text, "html.parser")
                cal_link = soup.find("a", href=lambda h: h and h.endswith("_cal.xml"))
                if cal_link:
                    cal_file = cal_link["href"].split("/")[-1]
            
            if not cal_link:
                for file in root.findall("InputFiles/File"):
                    if file.text and file.text.endswith("_cal.xml"):
                        cal_file = file.text
                        break
            
            if not cal_file:
                for file in root.findall("InputFiles/File"):
                    if file.text and file.text.endswith("xsd"):
                        cal_file = file.text
                        break

            
            if cal_file:
                response = requests.get(urlNoFile+cal_file, impersonate="chrome101", stream=False,timeout=30)
                response.raise_for_status()               # Check for download errors
                saveFilePath = os.path.join(savePath, f"{cik}_{accessionNumber}_{cal_file}")
                Path(saveFilePath).write_bytes(response.content)
                return saveFilePath
            else:
                print(f"No _cal.xml file found {url}")

        print(f"Files to download: {len(df)}")
        count = 1

        for index, row in df.iterrows():
            try:
                file = download_cal_xmls(row['url'], row['url_no_file'], row['cik'], row['accessionNumber'], save_path)
                if file:
                    print(f"Downloaded: {file} (Count: {count})")
                    count += 1
            except Exception as e:
                continue

load_dotenv()
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT") or Path(__file__).resolve().parent.parent)
DATA_DIR = PROJECT_ROOT / "Data"
db_directory = DATA_DIR
db_path = os.path.join(db_directory, "secFilingsDb.duckdb")

conn = ddb.connect(db_path, read_only=True)
download_cal_xmls(conn)

FileNotFoundError: [Errno 2] No such file or directory